# Communicating with Non-Technical Stakeholders

**Phase 11** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-11/10-communicating-with-stakeholders.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You are two weeks from launch. Your weekly status update says: "We completed integration of the hybrid retrieval module with cross-encoder reranking. RAGAS scores improved: faithfulness 0.82 -> 0.89, answer relevance 0.77 -> 0.85. Latency p95 is 1.4s. Still investigating occasional hallucinations on edge cases."

Your primary stakeholder, a VP of Operations, forwards this to the CFO with the message: "The AI team says the system is sometimes hallucinating. Should we be concerned?"

The CFO emails the CEO. The CEO asks for a meeting.

You caused a crisis by sending an accurate technical update. The problem was not the content; it was the audience. The VP read "hallucinations on edge cases" as...

## The Concept

### The Three Translation Problems

```
TRANSLATION PROBLEM 1: METRIC TRANSLATION
  Technical: "RAGAS faithfulness improved from 0.82 to 0.89"
  Stakeholder hears: "They changed a number. I do not know if that is good."
  Translation: "The system now gives accurate, well-sourced answers 89% of the time,
                up from 82%. The improvement means fewer corrections needed by your team."

TRANSLATION PROBLEM 2: UNCERTAINTY TRANSLATION
  Technical: "Occasional hallucinations on edge cases under investigation"
  Stakeholder hears: "The system makes things up and they do not know when or why...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
StakeholderTranslator: rewrite technical AI project updates for
non-technical executive audiences, then score the output.

Usage:
    python main.py --demo
    python main.py --update "your technical update text here"
"""

import argparse
import json
import os
import sys
from dataclasses import dataclass

import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-3-5-haiku-20241022"


@dataclass
class TranslationResult:
    original: str
    translated: str
    clarity_score: float
    business_relevance_score: float
    jargon_score: float              # 0 = no jargon, 1 = full jargon
    improvements: list[str]


TRANSLATE_PROMPT = """You are an expert at translating technical AI project updates into clear, business-focused language for non-technical stakeholders.

The audience is a VP or C-level executive who:
- Has no background in machine learning or software engineering
- Cares about business outcomes: time saved, cost reduced, risk managed, users helped
- Makes decisions based on confidence and clarity, not technical details
- Will escalate or block a project if they feel uncertain or out of the loop

Technical update to translate:
{update}

Translate this into a stakeholder-ready update. Then score the original and your translation.

Return a JSON object:
{{
  "translation": "<your stakeholder-friendly rewrite of the update>",
  "clarity_score": <float 0.0-1.0: how clear is the translated version to a non-technical reader>,
  "business_relevance_score": <float 0.0-1.0: does it speak to business outcomes and impact?>,
  "jargon_score": <float 0.0-1.0: how much technical jargon remains? 0=none, 1=all jargon>,
  "improvements": ["<specific change 1>", "<specific change 2>", "<specific change 3>"]
}}

Translation rules:
- Replace every metric with its business meaning (e.g., "accuracy 0.84" becomes "correctly handles 84% of requests")
- Replace "hallucination" with a concrete description of what happens and how often
- Replace infrastructure terms (RAG, embedding, p95, tokens, RAGAS, F1, precision, recall) with plain-language equivalents
- Keep the update under 150 words
- End with either a concrete status (on track / at risk) or a specific ask
- Do not introduce uncertainty you did not have - be honest about risks using plain language
- Never start with "I hope this email finds you well" or similar filler

Return only the JSON object."""


def translate_update(technical_update: str) -> TranslationResult:
    """Translate a technical update into stakeholder-ready language and score it."""
    prompt = TRANSLATE_PROMPT.format(update=technical_update)

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )

    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        lines = raw.split("\n")
        raw = "\n".join(lines[1:-1])

    data = json.loads(raw)

    return TranslationResult(
        original=technical_update,
        translated=data["translation"],
        clarity_score=float(data["clarity_score"]),
        business_relevance_score=float(data["business_relevance_score"]),
        jargon_score=float(data["jargon_score"]),
        improvements=data.get("improvements", []),
    )


def print_translation(result: TranslationResult) -> None:
    """Print the original, translation, and scores."""
    print("\n" + "=" * 60)
    print("STAKEHOLDER TRANSLATOR")
    print("=" * 60)

    print("\n--- ORIGINAL (TECHNICAL) ---")
    print(result.original)

    print("\n--- TRANSLATED (STAKEHOLDER-READY) ---")
    print(result.translated)

    print("\n--- SCORES ---")
    print(f"Clarity:            {result.clarity_score:.2f} / 1.0")
    print(f"Business relevance: {result.business_relevance_score:.2f} / 1.0")
    print(f"Jargon remaining:   {result.jargon_score:.2f} / 1.0  (lower is better)")

    if result.improvements:
        print("\n--- KEY CHANGES MADE ---")
        for improvement in result.improvements:
            print(f"  - {improvement}")

    avg_score = (
        result.clarity_score
        + result.business_relevance_score
        + (1.0 - result.jargon_score)
    ) / 3

    print(f"\nOverall translation quality: {avg_score:.2f} / 1.0")
    if avg_score >= 0.80:
        print("STATUS: Ready to send to stakeholders.")
    elif avg_score >= 0.65:
        print("STATUS: Review and refine before sending.")
    else:
        print("STATUS: Needs significant rework.")
    print("=" * 60 + "\n")


DEMO_UPDATES = [
    {
        "name": "Status Update 1: Technical Progress",
        "text": (
            "Completed integration of the hybrid retrieval module with cross-encoder reranking. "
            "RAGAS scores improved: faithfulness 0.82 -> 0.89, answer relevance 0.77 -> 0.85. "
            "Latency p95 is 1.4s, within SLA. Still investigating occasional hallucinations "
            "on edge cases involving multi-hop reasoning over documents > 50 pages."
        ),
    },
    {
        "name": "Status Update 2: Risk Escalation",
        "text": (
            "We identified a risk: the customer's Oracle CRM database does not have an API. "
            "We need to build an ETL pipeline to export data nightly to S3 as a workaround. "
            "This adds approximately 8 engineering days and introduces a data freshness "
            "constraint: the system will only have access to data as of the previous day's "
            "export. This may impact the real-time classification use case."
        ),
    },
    {
        "name": "Status Update 3: Pilot Results",
        "text": (
            "Pilot results for the 4-week support ticket classification evaluation: "
            "precision 0.87, recall 0.83, F1 0.85. Escalation rate dropped from 34% to 18%. "
            "Average time-to-resolution improved from 185s to 62s. "
            "Correlation between model confidence score and task success: 0.71. "
            "Token usage averaging 2,800 tokens per query, within budget."
        ),
    },
]


def run_demo() -> None:
    """Run the translator on three demo technical updates."""
    for item in DEMO_UPDATES:
        print(f"\n{'#' * 60}")
        print(f"  {item['name']}")
        print(f"{'#' * 60}")
        result = translate_update(item["text"])
        print_translation(result)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="StakeholderTranslator: rewrite technical AI updates for executive audiences"
    )
    parser.add_argument(
        "--demo",
        action="store_true",
        help="Run on 3 built-in demo technical updates",
    )
    parser.add_argument(
        "--update",
        help="Translate a single technical update (pass as a string)",
    )
    args = parser.parse_args()

    if args.demo:
        run_demo()
    elif args.update:
        result = translate_update(args.update)
        print_translation(result)
    else:
        print("Usage:")
        print("  python main.py --demo")
        print("  python main.py --update 'your technical update text'")
        sys.exit(1)

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What was the root cause of this communication failure?**

- A. The RAGAS score is too low and should be higher before sharing with stakeholders
- B. The VP should not have forwarded the update without asking for clarification first
- C. The update used technical terms ('hallucinations', 'RAGAS', 'p95') that a non-technical reader interpreted negatively without the context that these are normal and managed aspects of AI systems
- D. The update was too long and should have been a single-sentence summary

**2. Which of these is the most effective opening sentence for the results summary?**

- A. 'The AI system achieved an F1 score of 0.85 in the 4-week pilot.'
- B. 'During the 4-week pilot, the AI system reduced the percentage of support tickets requiring escalation to senior agents from 34% to 18%, and cut average resolution time from 3 minutes to just over 1 minute.'
- C. 'Pilot metrics show significant improvement across all measured dimensions with p-values below 0.05.'
- D. 'Our model performed well in the pilot with strong precision and recall numbers.'

**3. Which is the most effective response?**

- A. 'The system has a 15% error rate on our test set, which is industry standard for this type of AI.'
- B. 'All AI systems make mistakes. The important thing is that our error rate is low.'
- C. 'The system answers with high confidence on 85% of queries. For the remaining 15%, it flags the answer for human review rather than guessing - so your team only sees answers the system is confident about. The unsure ones go to a queue for agent review.'
- D. 'The hallucination rate is approximately 3% on edge cases, which is acceptable given the use case.'

**4. Which element is most critical to include in the escalated version?**

- A. A technical diagram of the ETL pipeline architecture
- B. A specific ask with a deadline: the decision the VP needs to make, what their options are, and when you need the answer
- C. The full technical explanation of why the Oracle CRM does not have an API
- D. A comparison of ETL pipeline vendors and their costs

**5. What is the most likely issue, and what should you revise?**

- A. The translation is good enough to send as-is
- B. The translation is clear but still contains some jargon and is not fully grounded in business outcomes; review for remaining technical terms and ensure every metric has a business consequence attached
- C. The translation failed - a quality score of 0.75 means it should be discarded
- D. The jargon score of 0.31 means 31% of the text is jargon, which is acceptable for a technical audience

**6. How do you respond to your manager's feedback?**

- A. Agree and restore the technical terms to show respect for the executives' intelligence
- B. Explain that the issue is not intelligence but vocabulary and context: a CEO who does not know what RAGAS is will not ask for clarification, they will make decisions based on incomplete interpretation
- C. Find a middle ground: use half the technical terms and translate the other half
- D. Ask the executives directly whether they want technical updates

**7. What color should this update be, and what must it include?**

- A. GREEN - the system itself is on track and the IT dependency is not your problem
- B. YELLOW - a specific risk exists that may affect the launch date; the update must name the risk, the impact, what you are doing, and what you need from the stakeholder (likely: their help expediting the IT ticket)
- C. RED - any dependency risk automatically makes a launch red status
- D. YELLOW - but only mention it if the delay becomes certain

**8. How would you rewrite the core of this sentence for a stakeholder audience?**

- A. 'The AI's search component was updated to better understand your company's specific vocabulary and terminology, improving its ability to find relevant information from your documents.'
- B. 'The embedding model was fine-tuned on your data, which improved performance metrics on domain-specific queries.'
- C. 'Technical improvements were made to the AI system this week.'
- D. 'The AI system is now smarter after we updated it with your company's data.'

_Answers are in `checks.json` in the lesson directory._